In [38]:
import pandas as pd

Umieścić w folderze "../data/movieLens" pobrane dane ze strony(rozpakowane)

title_movies_merged.csv to ramka danych powstała w wyniku poprzedniego łączenia (to co Gabi robiła) worldwide-box z imdb movies. Polecam zapisany plik.csv w folderze "../data/merged/title_movies_merged.csv"

## MOVIE LENS

Z tej bazy przede wszystkim będziemy potrzebować links.csv (ma id do innych też baz) oraz movies.csv (z tytułami, rokiem i gatunkiem) <br><br>

W innych plikach są ratingi osobnych użytkowników oraz słowa, którymi użytkownicy określali film (ale nie zbiorczo, ale osobno dane każdego użytkownika). Jest też tabela, gdzie określana jest trafność tych określeń ludzi. Można wybrać i przypisać do każdego fimu np 5 najbardziej trafnych określeń

In [39]:
links = pd.read_csv("../data/movieLens/links.csv", dtype={'imdbId': str, 'tmdbId' : str})
links

,movieId,imdbId,tmdbId
0,1,0114709,862
1,2,0113497,8844
2,3,0113228,15602
3,4,0114885,31357
4,5,0113041,11862
...,...,...,...
86532,288967,14418234,845861
86533,288971,11162178,878958
86534,288975,0070199,150392
86535,288977,23050520,1102551


In [40]:
mmovies = pd.read_csv("../data/movieLens/movies.csv")
mmovies.head(10)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [41]:
mmovies = pd.merge(
    mmovies,
    links,
    on = 'movieId',
    suffixes=('_movies', '_links')
)

In [42]:
mmovies.head(10)

,movieId,title,genres,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,0114709,862
1,2,Jumanji (1995),Adventure|Children|Fantasy,0113497,8844
2,3,Grumpier Old Men (1995),Comedy|Romance,0113228,15602
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,0114885,31357
4,5,Father of the Bride Part II (1995),Comedy,0113041,11862
5,6,Heat (1995),Action|Crime|Thriller,0113277,949
6,7,Sabrina (1995),Comedy|Romance,0114319,11860
7,8,Tom and Huck (1995),Adventure|Children,0112302,45325
8,9,Sudden Death (1995),Action,0114576,9091
9,10,GoldenEye (1995),Action|Adventure|Thriller,0113189,710


In [43]:
mmovies['year'] = mmovies['title'].str.extract(r'\((\d{4})\)')
mmovies['title'] = mmovies['title'].str.replace(r'\s\(\d{4}\)', '', regex=True)
mmovies['year'] = pd.to_numeric(mmovies['year'], errors='coerce')

In [44]:
mmovies = mmovies[['movieId', 'imdbId', 'tmdbId', 'title', 'year', 'genres']]

In [45]:
mmovies.head(10)

,movieId,imdbId,tmdbId,title,year,genres
0,1,0114709,862,Toy Story,1995.0,Adventure|Animation|Children|Comedy|Fantasy
1,2,0113497,8844,Jumanji,1995.0,Adventure|Children|Fantasy
2,3,0113228,15602,Grumpier Old Men,1995.0,Comedy|Romance
3,4,0114885,31357,Waiting to Exhale,1995.0,Comedy|Drama|Romance
4,5,0113041,11862,Father of the Bride Part II,1995.0,Comedy
5,6,0113277,949,Heat,1995.0,Action|Crime|Thriller
6,7,0114319,11860,Sabrina,1995.0,Comedy|Romance
7,8,0112302,45325,Tom and Huck,1995.0,Adventure|Children
8,9,0114576,9091,Sudden Death,1995.0,Action
9,10,0113189,710,GoldenEye,1995.0,Action|Adventure|Thriller


Próbuję jeszcze połączyć z najważniejszymi słowami kluczowymi

In [46]:
genome_scores = pd.read_csv("../data/movieLens/genome-scores.csv")
genome_tags = pd.read_csv('../data/movieLens/genome-tags.csv')

In [47]:
scores_with_names = pd.merge(genome_scores, genome_tags, on='tagId')
scores_with_names = scores_with_names.sort_values(by=['movieId', 'relevance'], ascending=[True, False])
top5_tags_long = scores_with_names.groupby('movieId').head(5)
top5_tags_final = top5_tags_long.groupby('movieId')['tag'].apply(lambda x: ', '.join(x)).reset_index()
top5_tags_final.rename(columns={'tag': 'top_5_tags'}, inplace=True)

In [48]:
top5_tags_final.head(10)

,movieId,top_5_tags
0,1,"toys, computer animation, pixar animation, ani..."
1,2,"adventure, children, fantasy, kids, childhood"
2,3,"good sequel, sequel, sequels, comedy, original"
3,4,"women, chick flick, girlie movie, romantic, di..."
4,5,"good sequel, sequel, sequels, midlife crisis, ..."
5,6,"crime, heist, imdb top 250, great acting, action"
6,7,"remake, romantic, romance, romantic comedy, go..."
7,8,"runaway, adapted from:book, based on a book, n..."
8,9,"action, good action, lone hero, hostage, actio..."
9,10,"007 (series), 007, bond, action, franchise"


In [49]:
movieLens = pd.merge(
    mmovies,
    top5_tags_final,
    on = 'movieId',
    how = "left"
)

In [50]:
movieLens.head(10)

,movieId,imdbId,tmdbId,title,year,genres,top_5_tags
0,1,0114709,862,Toy Story,1995.0,Adventure|Animation|Children|Comedy|Fantasy,"toys, computer animation, pixar animation, ani..."
1,2,0113497,8844,Jumanji,1995.0,Adventure|Children|Fantasy,"adventure, children, fantasy, kids, childhood"
2,3,0113228,15602,Grumpier Old Men,1995.0,Comedy|Romance,"good sequel, sequel, sequels, comedy, original"
3,4,0114885,31357,Waiting to Exhale,1995.0,Comedy|Drama|Romance,"women, chick flick, girlie movie, romantic, di..."
4,5,0113041,11862,Father of the Bride Part II,1995.0,Comedy,"good sequel, sequel, sequels, midlife crisis, ..."
5,6,0113277,949,Heat,1995.0,Action|Crime|Thriller,"crime, heist, imdb top 250, great acting, action"
6,7,0114319,11860,Sabrina,1995.0,Comedy|Romance,"remake, romantic, romance, romantic comedy, go..."
7,8,0112302,45325,Tom and Huck,1995.0,Adventure|Children,"runaway, adapted from:book, based on a book, n..."
8,9,0114576,9091,Sudden Death,1995.0,Action,"action, good action, lone hero, hostage, actio..."
9,10,0113189,710,GoldenEye,1995.0,Action|Adventure|Thriller,"007 (series), 007, bond, action, franchise"


In [51]:
movieLens.shape

(86537, 7)

In [52]:
movieLens.to_csv("../data/merged/movieLens.csv", index=False)